In [2]:
import re
import time
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from collections import deque

In [3]:
SEEDS = [
    "https://www.google.com",
    "https://www.wikipedia.org",
    # add more seed URLs here
]

In [4]:
MAX_PAGES = 200        # safety limit – increase slowly, never "infinite"
REQUEST_TIMEOUT = 5    # seconds
CRAWL_DELAY = 0.5      # seconds between requests


In [5]:
# Regex to find .ir domains
IR_DOMAIN_RE = re.compile(
    r'(?:https?://)?(?:[a-zA-Z0-9-]+\.)+[a-zA-Z0-9-]+\.ir\b',
    re.IGNORECASE
)


In [6]:
queue = deque(SEEDS)
visited_urls = set()
found_ir_domains = set()


def extract_links(base_url, html):
    """Extract absolute links from HTML."""
    soup = BeautifulSoup(html, "html.parser")
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()

        # Skip mailto:, javascript:, etc.
        if href.startswith("#") or href.startswith("mailto:") or href.startswith("javascript:"):
            continue

        # Make absolute URL
        absolute = urljoin(base_url, href)
        parsed = urlparse(absolute)

        # We only want http/https
        if parsed.scheme in ("http", "https"):
            links.add(absolute)

    return links


def extract_ir_domains(text):
    """Find all .ir domains in a text."""
    matches = IR_DOMAIN_RE.findall(text)
    return {m.lower() for m in matches}


pages_crawled = 0

while queue and pages_crawled < MAX_PAGES:
    url = queue.popleft()

    if url in visited_urls:
        continue
    visited_urls.add(url)

    try:
        print(f"[{pages_crawled+1}/{MAX_PAGES}] Fetching:", url)
        resp = requests.get(url, timeout=REQUEST_TIMEOUT, headers={
            "User-Agent": "SimpleEducationalCrawler/1.0"
        })
    except Exception as e:
        print("  !! Request failed:", e)
        continue

    if resp.status_code != 200 or "text/html" not in resp.headers.get("Content-Type", ""):
        continue

    html = resp.text
    pages_crawled += 1

    # 1) Extract .ir domains from the page content
    new_domains = extract_ir_domains(html)
    if new_domains:
        print("  -> Found .ir domains:", new_domains)
        found_ir_domains.update(new_domains)

    # 2) Extract new links to crawl
    links = extract_links(url, html)
    for link in links:
        if link not in visited_urls:
            queue.append(link)

    # 3) Be nice – do not hammer servers
    time.sleep(CRAWL_DELAY)

print("\n=== Crawl finished ===")
print(f"Crawled {pages_crawled} pages.")
print(f"Found {len(found_ir_domains)} unique .ir domains:")

for d in sorted(found_ir_domains):
    print(" -", d)

[1/200] Fetching: https://www.google.com
[2/200] Fetching: https://www.wikipedia.org
[3/200] Fetching: https://play.google.com/?hl=en&tab=w8
[4/200] Fetching: https://www.google.com/intl/en/policies/privacy/
[5/200] Fetching: https://www.google.com/intl/en/ads/
[6/200] Fetching: https://drive.google.com/?tab=wo
  -> Found .ir domains: {'v.country.ir'}
[7/200] Fetching: https://www.google.com/preferences?hl=en
[7/200] Fetching: https://www.google.com/intl/en/policies/terms/
[8/200] Fetching: https://maps.google.com/maps?hl=en&tab=wl
[9/200] Fetching: https://www.google.com/imghp?hl=en&tab=wi
[10/200] Fetching: https://mail.google.com/mail/?tab=wm
  -> Found .ir domains: {'v.country.ir'}
[11/200] Fetching: https://news.google.com/?tab=wn
[12/200] Fetching: https://www.google.com/intl/en/about.html
[13/200] Fetching: https://www.google.com/intl/en/about/products?tab=wh
[14/200] Fetching: https://www.youtube.com/?tab=w1
[15/200] Fetching: http://www.google.com/history/optout?hl=en
[16/200]

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
check_english_ir_domains.py

From English words, form <word>.ir, validate with regex, check DNS existence (A/AAAA by default).
Works both from CLI and inside Jupyter (ignores injected -f/--f args, avoids stdin blocking in notebooks).

Examples (CLI):
  python3 check_english_ir_domains.py --from-text notes.txt --concurrency 25 --verbose
  python3 check_english_ir_domains.py --from-wordlist words.txt --min-len 4 --max-len 15 --out-csv found.csv
  echo "Cloudflare and digikala are popular" | python3 check_english_ir_domains.py --verbose
  python3 -m pip install dnspython && \
  python3 check_english_ir_domains.py --from-text corpus.txt --dns-record MX --concurrency 10 --verbose
"""

import argparse
import csv
import json
import re
import socket
import sys
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Iterable, Optional, Tuple

# Optional import for MX/NS/ANY queries
try:
    import dns.resolver  # type: ignore
    HAS_DNSPYTHON = True
except Exception:
    HAS_DNSPYTHON = False

# ------------------------------ Regex / rules ------------------------------ #

# English word tokens only (letters A–Z). We’ll lowercase them.
WORD_RE = re.compile(r"[A-Za-z]+")

# Single DNS label (ASCII letters/digits, hyphens not at edges, max 63 chars)
LABEL_RE = re.compile(r"^[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?$")

def valid_label(label: str) -> bool:
    """Check if a string is a valid ASCII DNS label."""
    return bool(LABEL_RE.match(label))

def make_domain_from_word(word: str) -> Optional[str]:
    """Create <word>.ir if 'word' is a valid label; else return None."""
    w = word.lower()
    if not w or not valid_label(w):
        return None
    return f"{w}.ir"

def extract_english_words(text: str) -> Iterable[str]:
    """Yield lowercase English-only word tokens from text."""
    for m in WORD_RE.finditer(text):
        yield m.group(0).lower()

# ------------------------------ DNS helpers -------------------------------- #

def dns_exists(domain: str, record: str = "A", timeout: float = 3.0) -> Tuple[bool, str, list[str]]:
    """
    Check if domain exists via DNS.
      record='A'|'AAAA' uses socket.getaddrinfo (no extra deps).
      record in {'MX','NS','ANY'} uses dnspython if available.
    Returns (exists:boolean, method:str, ips:list[str]).
    """
    record = record.upper()
    if record in ("A", "AAAA"):
        family = socket.AF_INET if record == "A" else socket.AF_INET6
        try:
            infos = socket.getaddrinfo(domain, None, family, 0, 0, socket.AI_ADDRCONFIG)
            ips = sorted({addr[0] for _fam, _stype, _proto, _canon, addr in infos})
            return True, record, ips
        except socket.gaierror as e:
            return False, f"{record}:{e.__class__.__name__}", []
        except Exception as e:
            return False, f"{record}:{type(e).__name__}", []

    if not HAS_DNSPYTHON:
        return False, f"{record}:dnspython_not_installed", []

    try:
        if record == "ANY":
            for rrtype in ("A", "AAAA", "MX", "NS"):
                try:
                    ans = dns.resolver.resolve(domain, rrtype, lifetime=timeout, raise_on_no_answer=False)
                    if getattr(ans, "rrset", None):
                        return True, f"ANY:{rrtype}", [str(r) for r in ans]
                except Exception:
                    pass
            return False, "ANY:none", []
        else:
            ans = dns.resolver.resolve(domain, record, lifetime=timeout, raise_on_no_answer=False)
            if getattr(ans, "rrset", None):
                return True, record, [str(r) for r in ans]
            return False, f"{record}:noanswer", []
    except dns.resolver.NXDOMAIN:
        return False, "NXDOMAIN", []
    except dns.resolver.NoNameservers:
        return False, "NoNameservers", []
    except dns.resolver.Timeout:
        return False, "Timeout", []
    except Exception as e:
        return False, f"{type(e).__name__}", []

# ------------------------------- I/O helpers -------------------------------- #

def load_candidates_from_text(path: Path, min_len: int, max_len: int, unique: bool):
    text = path.read_text(encoding="utf-8", errors="ignore")
    words = (w for w in extract_english_words(text) if min_len <= len(w) <= max_len)
    domains = []
    for w in words:
        d = make_domain_from_word(w)
        if d:
            domains.append(d)
    return sorted(set(domains)) if unique else domains

def load_candidates_from_string(text: str, min_len: int, max_len: int, unique: bool):
    words = (w for w in extract_english_words(text) if min_len <= len(w) <= max_len)
    domains = []
    for w in words:
        d = make_domain_from_word(w)
        if d:
            domains.append(d)
    return sorted(set(domains)) if unique else domains

def load_candidates_from_wordlist(path: Path, min_len: int, max_len: int, unique: bool):
    domains = []
    with path.open("r", encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            w = line.strip()
            if not w:
                continue
            if not WORD_RE.fullmatch(w):
                continue
            w = w.lower()
            if not (min_len <= len(w) <= max_len):
                continue
            d = make_domain_from_word(w)
            if d:
                domains.append(d)
    return sorted(set(domains)) if unique else domains

def running_in_ipykernel() -> bool:
    """Detect if running under IPython/Jupyter kernel to avoid stdin blocking."""
    try:
        from IPython import get_ipython  # type: ignore
        ip = get_ipython()
        return ip is not None
    except Exception:
        return False

# --------------------------------- main ------------------------------------ #

def main():
    parser = argparse.ArgumentParser(
        description="Iterate English words, form <word>.ir, validate with regex, and check if the domain exists.",
        allow_abbrev=False,  # prevent '-f' matching '--from-text/--from-wordlist'
    )
    src = parser.add_mutually_exclusive_group(required=False)
    src.add_argument("--from-text", type=Path, help="Path to a text file to mine English words from.")
    src.add_argument("--from-wordlist", type=Path, help="Path to a newline-delimited English wordlist (A–Z only).")
    parser.add_argument("--text", type=str, help="Raw English text to scan (alternative to --from-text/--from-wordlist).")
    parser.add_argument("--min-len", type=int, default=3, help="Minimum word length to consider (default: 3).")
    parser.add_argument("--max-len", type=int, default=63, help="Maximum word length to consider (≤63, default: 63).")
    parser.add_argument("--concurrency", type=int, default=20, help="Number of parallel DNS checks (default: 20).")
    parser.add_argument("--dns-record", choices=["A", "AAAA", "MX", "NS", "ANY"], default="A",
                        help="Record type to test existence (default: A). MX/NS/ANY require dnspython.")
    parser.add_argument("--timeout", type=float, default=3.0, help="DNS query timeout seconds (default: 3.0).")
    parser.add_argument("--sleep", type=float, default=0.0, help="Optional per-task sleep to be polite (seconds).")
    parser.add_argument("--stop-on", type=int, default=0, help="Stop after finding this many LIVE domains (0 = check all).")
    parser.add_argument("--out-csv", type=Path, help="Write results to CSV file.")
    parser.add_argument("--out-json", type=Path, help="Write results to JSON file.")
    parser.add_argument("--no-unique", action="store_true", help="Do not de-duplicate candidate words.")
    parser.add_argument("--verbose", action="store_true", help="Print every check and running stats.")
    parser.add_argument("--live-only", action="store_true", help="Only print final LIVE domains list (still prints summary).")

    # Use parse_known_args to ignore Jupyter's '-f/--f' and other injected args
    args, _unknown = parser.parse_known_args()

    unique = not args.no_unique
    if args.max_len > 63 or args.max_len < 1:
        parser.error("--max-len must be between 1 and 63")

    # Load candidates
    candidates = []
    if args.from_wordlist:
        candidates = load_candidates_from_wordlist(args.from_wordlist, args.min_len, args.max_len, unique)
    elif args.from_text:
        candidates = load_candidates_from_text(args.from_text, args.min_len, args.max_len, unique)
    elif args.text is not None:
        candidates = load_candidates_from_string(args.text, args.min_len, args.max_len, unique)
    else:
        # Only read stdin if not in an interactive IPython/Jupyter session
        if not sys.stdin.isatty() and not running_in_ipykernel():
            data = sys.stdin.read()
            if data.strip():
                candidates = load_candidates_from_string(data, args.min_len, args.max_len, unique)
        if not candidates:
            demo = "We saw Google and Amazon; also Cloudflare, Pars, Giti, DigiKala. Random words: cat dog run fly."
            print("(No input provided; scanning demo text)\n", file=sys.stderr)
            candidates = load_candidates_from_string(demo, args.min_len, args.max_len, unique)

    total = len(candidates)
    if total == 0:
        print("No candidate English words found.")
        return

    print(f"Loaded {total} candidate words ⇒ {total} potential .ir domains.")
    if args.dns_record in {"MX", "NS", "ANY"} and not HAS_DNSPYTHON:
        print("[warn] dnspython not installed; MX/NS/ANY checks will be downgraded to A/AAAA.", file=sys.stderr)

    start_ts = time.time()
    results = []
    counts = Counter()
    live_found = 0

    def worker(domain: str):
        if args.sleep > 0:
            time.sleep(args.sleep)
        exists, method, ips = dns_exists(domain, record=args.dns_record, timeout=args.timeout)
        return {"domain": domain, "exists": bool(exists), "method": method, "ips": ips}

    from concurrent.futures import ThreadPoolExecutor, as_completed
    processed = 0
    with ThreadPoolExecutor(max_workers=args.concurrency) as ex:
        futures = {ex.submit(worker, d): d for d in candidates}
        try:
            for fut in as_completed(futures):
                res = fut.result()
                results.append(res)
                processed += 1
                status = "LIVE" if res["exists"] else "ABSENT"
                counts[status] += 1

                if args.verbose:
                    ips_s = f" -> {','.join(res['ips'])}" if res["ips"] else ""
                    print(f"[{processed}/{total}] {status:6} {res['domain']}  ({res['method']}){ips_s}")

                if args.stop_on and res["exists"]:
                    live_found += 1
                    if live_found >= args.stop_on:
                        for other in futures:
                            if not other.done():
                                other.cancel()
                        break
        except KeyboardInterrupt:
            print("\n[!] Interrupted by user. Finishing up…", file=sys.stderr)

    # Sort: LIVE first then alphabetically
    results.sort(key=lambda r: (0 if r["exists"] else 1, r["domain"]))

    # Print all checks unless --live-only
    if not args.live_only:
        print("\n=== All checks ===")
        for r in results:
            tag = "LIVE " if r["exists"] else "ABSENT"
            ips_s = f"  [{', '.join(r['ips'])}]" if r["exists"] and r["ips"] else ""
            print(f"{tag:6} {r['domain']:<30} ({r['method']}){ips_s}")

    # Print LIVE list
    print("\n=== LIVE .ir domains ===")
    live = [r for r in results if r["exists"]]
    if live:
        for r in live:
            ips_s = f"  [{', '.join(r['ips'])}]" if r["ips"] else ""
            print(f"- {r['domain']}{ips_s}")
    else:
        print("(none)")

    # Summary
    elapsed = time.time() - start_ts
    dur = max(1e-9, elapsed)
    qps = processed / dur
    print("\n=== Summary ===")
    print(f"Checked      : {processed} / {total}")
    print(f"Duration     : {dur:.1f}s  (~{qps:.2f} qps)")
    print(f"LIVE         : {len(live)}")
    print(f"ABSENT       : {counts['ABSENT']}")
    if args.stop_on:
        print(f"Stopped after reaching --stop-on={args.stop_on} LIVE domains.")

    # Exports
    if args.out_csv:
        with args.out_csv.open("w", newline="", encoding="utf-8") as fh:
            writer = csv.DictWriter(fh, fieldnames=["domain", "exists", "method", "ips"])
            writer.writeheader()
            for r in results:
                row = r.copy()
                row["ips"] = ";".join(r["ips"])
                writer.writerow(row)
        print(f"Wrote CSV → {args.out_csv}")

    if args.out_json:
        with args.out_json.open("w", encoding="utf-8") as fh:
            json.dump(results, fh, ensure_ascii=False, indent=2)
        print(f"Wrote JSON → {args.out_json}")


if __name__ == "__main__":
    main()



=== Matches (.ir tokens found) ===
  1. ABSENT shop.digikala.ir               (A:gaierror)

=== Unique LIVE .ir domains ===
(none)

=== Summary ===
Tokens scanned : 9 / 9
LIVE matches   : 0 (unique: 0)
ABSENT matches : 1
Duration       : 0.0s  (~657.25 tokens/s)


(No input provided; using demo text)

